# 08 — Trifactorial HRV analysis

**Study question.** For each HRV metric separately, how are three demographic/hemodynamic/anthropometric factors jointly associated with the outcome, and does the interaction of two factors depend on the third factor?

This notebook extends Notebook 07 with **four factor triplets × seven HRV responses = 28 categorical analyses** and seven complementary continuous/mixed three-factor specifications per HRV response.

**Analysis conventions**

- **Original age, sex, BP and BMI categories remain intact** in the source data, descriptive outputs and faceted distribution plots. A new *analysis-only* age grouping (60–69/70+) complements the BP and BMI groupings already documented for Notebook 07, because original three-factor cells can have zero to two people. These groupings are **statistical**, not clinical classifications.
- **Primary categorical models** have all three main effects, all three two-factor interactions and the three-factor interaction. An additive model estimates adjusted main effects, a hierarchical second-order model tests two-factor interactions and the full model tests the third-order interaction. HC3-robust Wald tests are primary; hierarchical incremental partial R² is descriptive, not a p-value.
- **Conditional effects** are computed within levels of the other two factors, with nonparametric Cliff's delta, median differences, KS, Wasserstein, Brown–Forsythe and bootstrap intervals when both groups have at least 10 observations. Model-based conditional contrasts are also estimated using the full HC3 model.
- **Continuous/mixed models** complement, and do not replace, categorical analyses. Both SBP and DBP specifications are included separately, never together as independent BP factors.
- **Sensitivity** tests include outer 1% outcome trimming, an alternative outcome scale and original-age category feasibility. Low cell sizes and model diagnostics are explicitly flagged.
- The results are cross-sectional associations, not causal effects, diagnostic labels or prognostic outcomes. Mean RR and mean HR are physiologically/mathematically linked. VLF from five-minute HRV recordings warrants cautious interpretation.

**Dependencies:** pandas, numpy, scipy, statsmodels, patsy, matplotlib, seaborn, IPython. No new data-processing framework or external files are generated beyond the requested tables and figures.

## 0. Input data, factor definitions and paths

In [1]:
from pathlib import Path
from itertools import combinations
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from patsy import build_design_matrices, dmatrix
from IPython.display import display

INPUT_FILE = Path("../../outputs/dataset_final.csv")
OUTPUT_DIR = Path("../../eda_results/08_trifactorial_hrv_analysis")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
GLOBAL_DIR = FIGURE_DIR / "_global"
for directory in (TABLE_DIR, FIGURE_DIR, GLOBAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 20261125
BOOTSTRAP_REPLICATES = 350
MIN_CELL_N = 5
MIN_MODEL_CELL_N = 8
MIN_CONTRAST_N = 10
FDR_ALPHA = 0.05

HRV = {
    "t2m_pre_mean_rr": ("mean_rr", "Mean RR", "ms"),
    "t2m_pre_mean_hr": ("mean_hr", "Mean HR", "bpm"),
    "t2m_pre_sdnn": ("sdnn", "SDNN", "ms"),
    "t2m_pre_rmssd": ("rmssd", "RMSSD", "ms"),
    "t2m_pre_hf": ("hf", "HF", "ms²"),
    "t2m_pre_lf": ("lf", "LF", "ms²"),
    "t2m_pre_vlf": ("vlf", "VLF", "ms²"),
}
FACTORS = {
    "age_group": ["60-69", "70-79", "80+"],
    "sex_category": ["Female", "Male"],
    "bp_category": ["Optimal", "Normal", "High-normal", "Stage 1", "Stage 2", "Stage 3"],
    "bmi_category": ["Underweight", "Normal", "Overweight", "Obesity"],
}
FACTOR_LABELS = {"age_group":"Age", "sex_category":"Sex", "bp_category":"Blood pressure", "bmi_category":"BMI category"}
ANALYSIS_COL = {"age_group":"age_analysis", "sex_category":"sex_category", "bp_category":"bp_analysis", "bmi_category":"bmi_analysis"}
ANALYSIS_LEVELS = {
    "age_group": ["60-69", "70+"],
    "sex_category": ["Female", "Male"],
    "bp_category": ["Optimal/Normal", "High-normal", "Stage 1-3"],
    "bmi_category": ["Underweight/Normal", "Overweight", "Obesity"],
}
TRIPLETS = list(combinations(FACTORS, 3))
SHORT = {"age_group":"age", "sex_category":"sex", "bp_category":"bp", "bmi_category":"bmi"}
TRIPLET_NAMES = {triple:"_x_".join(SHORT[f] for f in triple) for triple in TRIPLETS}
PALETTES = {
    "age_group": dict(zip(FACTORS["age_group"], sns.color_palette("viridis", 3))),
    "sex_category": dict(zip(FACTORS["sex_category"], sns.color_palette("colorblind", 2))),
    "bp_category": dict(zip(FACTORS["bp_category"], sns.color_palette("crest", 6))),
    "bmi_category": dict(zip(FACTORS["bmi_category"], sns.color_palette("flare", 4))),
}
ANALYSIS_PALETTES = {
    "age_group": dict(zip(ANALYSIS_LEVELS["age_group"], sns.color_palette("viridis", 2))),
    "sex_category": dict(zip(ANALYSIS_LEVELS["sex_category"], sns.color_palette("colorblind", 2))),
    "bp_category": dict(zip(ANALYSIS_LEVELS["bp_category"], sns.color_palette("crest", 3))),
    "bmi_category": dict(zip(ANALYSIS_LEVELS["bmi_category"], sns.color_palette("flare", 3))),
}

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.facecolor":"white", "savefig.facecolor":"white",
                     "axes.spines.top":False, "axes.spines.right":False,
                     "axes.titleweight":"semibold", "grid.alpha":0.16})

df = pd.read_csv(INPUT_FILE)
original_shape = df.shape
required = list(HRV) + list(FACTORS) + ["age", "imc", "bp_systolic", "bp_diastolic"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns in final dataset: {missing}")
if df[required].isna().any().any():
    raise ValueError("Missing values in required HRV or factor columns; review preprocessing before continuing")
for col in list(HRV) + ["age", "imc", "bp_systolic", "bp_diastolic"]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(f"Expected numeric values in {col}")
for factor, levels in FACTORS.items():
    unexpected = set(df[factor].unique()) - set(levels)
    if unexpected:
        raise ValueError(f"Unexpected levels in {factor}: {sorted(unexpected)}")

# Analysis-only grouping; source CSV is not modified.
df["age_analysis"] = df.age_group.replace({"60-69":"60-69", "70-79":"70+", "80+":"70+"})
df["bp_analysis"] = df.bp_category.replace({"Optimal":"Optimal/Normal", "Normal":"Optimal/Normal",
    "High-normal":"High-normal", "Stage 1":"Stage 1-3", "Stage 2":"Stage 1-3", "Stage 3":"Stage 1-3"})
df["bmi_analysis"] = df.bmi_category.replace({"Underweight":"Underweight/Normal", "Normal":"Underweight/Normal"})
for factor in FACTORS:
    key = ANALYSIS_COL[factor]
    df[key] = pd.Categorical(df[key], categories=ANALYSIS_LEVELS[factor], ordered=True)

# Global HRV response scaling, matching Notebook 07.
SCALES = {}
for col in HRV:
    values = df[col]
    ratio = values.quantile(.99) / max(values.quantile(.01), 1e-12)
    SCALES[col] = "log10" if values.min() > 0 and ratio >= 30 else "identity"
model_scales = pd.DataFrame([{"hrv_variable":col,"hrv_name":HRV[col][1],"outcome_scale":SCALES[col],
    "q99_q01_ratio":df[col].quantile(.99)/max(df[col].quantile(.01),1e-12)} for col in HRV])
model_scales.to_csv(TABLE_DIR / "model_scales.csv", index=False)

for col, (slug, _label, _unit) in HRV.items():
    for triplet in TRIPLETS:
        (FIGURE_DIR / slug / TRIPLET_NAMES[triplet]).mkdir(parents=True, exist_ok=True)
    (FIGURE_DIR / slug / "_summary").mkdir(parents=True, exist_ok=True)

print(f"Participants: {len(df)} | HRV metrics: {len(HRV)} | factor triplets: {len(TRIPLETS)}")
print(f"Output folder: {OUTPUT_DIR}")
display(model_scales.round(3))

Participants: 476 | HRV metrics: 7 | factor triplets: 4
Output folder: ../../eda_results/08_trifactorial_hrv_analysis


,hrv_variable,hrv_name,outcome_scale,q99_q01_ratio
0,t2m_pre_mean_rr,Mean RR,identity,1.927
1,t2m_pre_mean_hr,Mean HR,identity,1.934
2,t2m_pre_sdnn,SDNN,log10,52.900
3,t2m_pre_rmssd,RMSSD,log10,34.118
4,t2m_pre_hf,HF,log10,1924.167
5,t2m_pre_lf,LF,log10,643.256
6,t2m_pre_vlf,VLF,log10,726.250


## 1. Full factor-triplet cell inventory

Both the **original** categories and **analysis-only** categories are counted, including **zero-count** combinations. An interaction can only be interpreted when the model matrix is estimable and the supporting cells are adequately populated.

In [2]:
def factor_grid(triplet, scheme):
    levels = [FACTORS[f] if scheme == "original" else ANALYSIS_LEVELS[f] for f in triplet]
    names = list(triplet) if scheme == "original" else [ANALYSIS_COL[f] for f in triplet]
    grid = pd.MultiIndex.from_product(levels, names=names).to_frame(index=False)
    counts = df.groupby(names, observed=True).size().rename("n").reset_index()
    return grid.merge(counts, on=names, how="left").fillna({"n":0}).assign(n=lambda x:x.n.astype(int))

support_tables = []
for triple in TRIPLETS:
    for scheme in ("original", "analysis"):
        grid = factor_grid(triple,scheme)
        for row in grid.itertuples(index=False):
            support_tables.append({"triplet":TRIPLET_NAMES[triple],"factor_a":triple[0],"factor_b":triple[1],
                "factor_c":triple[2],"category_scheme":scheme,
                "level_a":str(row[0]),"level_b":str(row[1]),"level_c":str(row[2]),
                "n":int(row.n),"empty":row.n==0,"n_lt5":row.n<5,"n_lt10":row.n<10})
support = pd.DataFrame(support_tables)
support.to_csv(TABLE_DIR / "factorial_cell_counts.csv",index=False)
support_summary = (support.groupby(["triplet","category_scheme"],as_index=False)
    .agg(total_cells=("n","size"),min_cell_n=("n","min"),empty_cells=("empty","sum"),
         cells_lt5=("n_lt5","sum"),cells_lt10=("n_lt10","sum"),participants=("n","sum")))
support_summary.to_csv(TABLE_DIR / "factorial_support_summary.csv",index=False)
assert support_summary.participants.eq(len(df)).all()
display(support_summary)

for triple in TRIPLETS:
    group = support.loc[(support.triplet==TRIPLET_NAMES[triple]) & (support.category_scheme=="original")]
    levels_c = FACTORS[triple[2]]
    ncols = min(3,len(levels_c)); nrows = int(np.ceil(len(levels_c)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(5.3*ncols,4.5*nrows),squeeze=False)
    for ax,lev_c in zip(axes.flat,levels_c):
        sub=group.loc[group.level_c==lev_c]
        matrix=sub.pivot(index="level_a",columns="level_b",values="n").reindex(index=FACTORS[triple[0]],columns=FACTORS[triple[1]])
        sns.heatmap(matrix,ax=ax,cmap="Blues",annot=True,fmt="d",linewidths=.6,
                    linecolor="white",cbar_kws={"label":"Participants"},annot_kws={"fontsize":9})
        ax.set(title=f"{FACTOR_LABELS[triple[2]]} = {lev_c}",xlabel=FACTOR_LABELS[triple[1]],ylabel=FACTOR_LABELS[triple[0]])
        ax.tick_params(axis="x",rotation=30)
    for ax in list(axes.flat)[len(levels_c):]:ax.axis("off")
    fig.suptitle(f"Original-category cell support | {TRIPLET_NAMES[triple]}",fontsize=17,fontweight="semibold")
    fig.tight_layout(rect=(0,0,1,.955),h_pad=2)
    fig.savefig(GLOBAL_DIR / f"{TRIPLET_NAMES[triple]}__cell_support.png",dpi=170,bbox_inches="tight")
    plt.close(fig)
print("Original-category support figures: 4")

,triplet,category_scheme,total_cells,min_cell_n,empty_cells,cells_lt5,cells_lt10,participants
0,age_x_bp_x_bmi,analysis,18,12,0,0,0,476
1,age_x_bp_x_bmi,original,72,0,17,44,50,476
2,age_x_sex_x_bmi,analysis,12,11,0,0,0,476
3,age_x_sex_x_bmi,original,24,0,1,9,10,476
4,age_x_sex_x_bp,analysis,12,13,0,0,0,476
5,age_x_sex_x_bp,original,36,0,3,12,20,476
6,sex_x_bp_x_bmi,analysis,18,9,0,0,2,476
7,sex_x_bp_x_bmi,original,48,0,9,19,28,476


findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.
findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Original-category support figures: 4


## 2. Distributional characterization by triplet

Descriptives use unmodified original levels. For each HRV variable and each observed factor combination, retain n, center, variability, quantiles, skewness and 1.5×IQR tail counts. Cells with no participants are still shown in the separate support inventory.

In [3]:
descriptive_rows = []
for col, (_slug, label, unit) in HRV.items():
    for triple in TRIPLETS:
        for scheme in ("original", "analysis"):
            keys = list(triple) if scheme == "original" else [ANALYSIS_COL[f] for f in triple]
            for values,part in df.groupby(keys,observed=True):
                sample=part[col].to_numpy(float)
                q1,q3=np.quantile(sample,[.25,.75]);iqr=q3-q1
                descriptive_rows.append({"triplet":TRIPLET_NAMES[triple],"category_scheme":scheme,
                    "hrv_variable":col,"hrv_label":label,"hrv_unit":unit,
                    "level_a":str(values[0]),"level_b":str(values[1]),"level_c":str(values[2]),
                    "n":len(sample),"mean":np.mean(sample),"median":np.median(sample),
                    "sd":np.std(sample,ddof=1) if len(sample)>1 else np.nan,
                    "q05":np.quantile(sample,.05),"q25":q1,"q75":q3,"q95":np.quantile(sample,.95),
                    "iqr":iqr,"min":np.min(sample),"max":np.max(sample),
                    "skewness":stats.skew(sample,bias=False) if len(sample)>=3 else np.nan,
                    "iqr_outliers":int(((sample<q1-1.5*iqr)|(sample>q3+1.5*iqr)).sum()) if len(sample)>=4 else np.nan,
                    "support_caution":len(sample)<MIN_CONTRAST_N})
descriptives=pd.DataFrame(descriptive_rows)
descriptives.to_csv(TABLE_DIR / "trifactorial_descriptives.csv",index=False)
print(f"Descriptive rows: {len(descriptives)}")
display(descriptives.head(12).round(3))

Descriptive rows: 1470


,triplet,category_scheme,hrv_variable,hrv_label,hrv_unit,level_a,level_b,level_c,n,mean,...,q05,q25,q75,q95,iqr,min,max,skewness,iqr_outliers,support_caution
0,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Female,High-normal,34,824.529,...,655.00,757.75,888.00,988.15,130.25,589.0,998.0,-0.170,0.0,False
1,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Female,Normal,38,864.079,...,713.35,793.25,921.75,1022.20,128.50,645.0,1296.0,1.257,1.0,False
2,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Female,Optimal,34,852.206,...,701.95,774.00,932.00,1007.40,158.00,617.0,1043.0,-0.063,0.0,False
3,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Female,Stage 1,29,824.862,...,675.60,760.00,875.00,1010.40,115.00,662.0,1046.0,0.327,0.0,False
4,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Female,Stage 2,13,904.923,...,732.40,809.00,1007.00,1044.80,198.00,730.0,1058.0,-0.377,0.0,False
5,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Female,Stage 3,2,640.500,...,594.15,614.75,666.25,686.85,51.50,589.0,692.0,NaN,NaN,True
6,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Male,High-normal,13,796.462,...,687.00,739.00,880.00,956.40,141.00,672.0,1005.0,0.938,0.0,False
7,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Male,Normal,11,857.364,...,732.00,810.50,935.00,956.50,124.50,704.0,962.0,-0.367,0.0,False
8,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Male,Optimal,7,914.571,...,804.10,837.00,956.00,1098.00,119.00,793.0,1140.0,1.263,1.0,True
9,age_x_sex_x_bp,original,t2m_pre_mean_rr,Mean RR,ms,60-69,Male,Stage 1,11,814.000,...,626.50,727.50,912.00,1028.50,184.50,592.0,1051.0,0.261,0.0,False


## 3. HC3 categorical models — adjusted main effects, pair interactions and three-way interactions

Factor terms use sum contrasts. Adjusted main effects and their incremental partial R² are tested in the **additive model**; two-factor interactions and their incremental partial R² are tested in a **hierarchical pair-interaction model** containing all main effects and all three two-factor terms; the three-factor interaction is tested in the **full model**. This avoids interpreting lower-order coefficient p-values from a three-way-interaction model as unconditional marginal associations.

The analytical categories are **age 60–69 vs 70+**, **sex Female/Male**, **BP Optimal/Normal vs High-normal vs Stage 1–3**, and **BMI Underweight/Normal vs Overweight vs Obesity**. A model is skipped if any factorial cell has fewer than eight observations, has a deficient design rank, or cannot produce a finite three-way Wald statistic. All failures are **recorded explicitly**; no silent results are manufactured.

In [4]:
def fitted_design_info(model, training_data):
    """Reconstruct and validate the Patsy design matrix used by a fitted model.

    Avoid relying on statsmodels internals such as model.model.data.design_info,
    which are absent in some versions and some fitted-data representations.
    """
    formula = model.model.formula
    if formula is None or "~" not in formula:
        raise ValueError("Expected a formula-based model to reconstruct its design matrix")
    rhs = formula.split("~", 1)[1]
    x = dmatrix(rhs, training_data, return_type="dataframe")
    if list(x.columns) != list(model.model.exog_names):
        raise ValueError("Patsy design columns differ from those of the fitted model")
    if not np.allclose(x.to_numpy(dtype=float), model.model.exog,
                       rtol=1e-10, atol=1e-10):
        raise ValueError("Reconstructed Patsy design differs from the fitted model")
    return x.design_info


def prediction_summary(model, design_info, observations, alpha=0.05):
    """Return predictions and HC3 confidence intervals without implicit Patsy lookup."""
    x = build_design_matrices([design_info], observations, return_type="dataframe")[0]
    if list(x.columns) != list(model.model.exog_names):
        raise ValueError("Prediction design columns do not match fitted model")
    return model.get_prediction(exog=x.to_numpy(dtype=float),
                                transform=False).summary_frame(alpha=alpha)


def wald_p(model, term):
    try:
        table = model.wald_test_terms(scalar=True).table
        pval = float(table.loc[term,"pvalue"])
        return pval if np.isfinite(pval) else np.nan
    except (KeyError,ValueError,AttributeError,np.linalg.LinAlgError):
        return np.nan

def fit_categorical(col, triple, data=None, scale=None):
    source=df if data is None else data
    a,b,c=[ANALYSIS_COL[f] for f in triple]
    sub=source[[a,b,c,col]].dropna().copy()
    scheme=scale or SCALES[col]
    if scheme=="log10" and (sub[col]<=0).any():
        return None,"nonpositive_log_outcome",np.nan
    sub["response"] = np.log10(sub[col]) if scheme=="log10" else sub[col].astype(float)
    support_grid=(sub.groupby([a,b,c],observed=True).size().reindex(
        pd.MultiIndex.from_product([ANALYSIS_LEVELS[f] for f in triple]),fill_value=0))
    min_cell=int(support_grid.min())
    if min_cell < MIN_MODEL_CELL_N:
        return None,"insufficient_cell_support",min_cell
    terms=[f"C({key}, Sum)" for key in (a,b,c)]
    add_formula="response ~ "+" + ".join(terms)
    pairs=[terms[0]+":"+terms[1],terms[0]+":"+terms[2],terms[1]+":"+terms[2]]
    pair_formula=add_formula+" + "+" + ".join(pairs)
    full_formula="response ~ "+" * ".join(terms)
    try:
        add=ols(add_formula,sub).fit(cov_type="HC3")
        second=ols(pair_formula,sub).fit(cov_type="HC3")
        full=ols(full_formula,sub).fit(cov_type="HC3")
    except (ValueError, np.linalg.LinAlgError):
        return None,"fit_error",min_cell
    if any(m.model.rank<m.model.exog.shape[1] for m in (add,second,full)):
        return None,"rank_deficient",min_cell
    triple_term=":".join(terms)
    triple_p=wald_p(full,triple_term)
    if not np.isfinite(triple_p):
        return None,"three_way_wald_unavailable",min_cell
    rss2=float(np.sum(second.resid**2));rss3=float(np.sum(full.resid**2))
    interaction_partial_r2=max(0.,(rss2-rss3)/rss2) if rss2>0 else np.nan
    full_design_info = fitted_design_info(full, sub)
    diagnostics={"condition_number":float(full.condition_number),
        "max_leverage":float(full.get_influence().hat_matrix_diag.max()),
        "max_cooks_distance":float(np.nanmax(full.get_influence().cooks_distance[0])),
        "parameters":int(full.model.exog.shape[1]),"rank":int(full.model.rank)}
    return {"add":add,"second":second,"full":full,"full_design_info":full_design_info,
            "terms":terms,"pair_terms":pairs,"triple_term":triple_term,
            "partial_r2":interaction_partial_r2,"diagnostics":diagnostics,
            "model_data":sub,"n":len(sub),"scheme":scheme},"ok",min_cell

categorical_records=[]
main_records=[]
second_records=[]
cat_fits={}
for col in HRV:
    for triple in TRIPLETS:
        name=TRIPLET_NAMES[triple]
        fit,status,min_cell=fit_categorical(col,triple)
        record={"hrv_variable":col,"triplet":name,"factor_a":triple[0],"factor_b":triple[1],
            "factor_c":triple[2],"n":len(df),"min_analysis_cell_n":min_cell,
            "model_scale":SCALES[col],"model_status":status,
            "low_support_warning":bool(min_cell<10)}
        if fit is not None:
            m=fit["full"]
            record.update({"three_way_p_hc3":wald_p(m,fit["triple_term"]),
                "three_way_partial_r2":fit["partial_r2"],"full_model_r2":float(m.rsquared),
                "full_model_adj_r2":float(m.rsquared_adj),"full_model_aic":float(m.aic),
                **fit["diagnostics"]})
            cat_fits[(col,name)]=fit
            rss_add=float(np.sum(fit["add"].resid**2))
            rss_second=float(np.sum(fit["second"].resid**2))
            for i,(factor,term) in enumerate(zip(triple,fit["terms"])):
                remaining=[t for j,t in enumerate(fit["terms"]) if j!=i]
                reduced=ols("response ~ "+" + ".join(remaining),fit["model_data"]).fit()
                rss_reduced=float(np.sum(reduced.resid**2))
                effect_size=max(0.,(rss_reduced-rss_add)/rss_reduced) if rss_reduced>0 else np.nan
                main_records.append({"hrv_variable":col,"triplet":name,"factor":factor,
                    "model":"additive_HC3","term":term,"p_hc3":wald_p(fit["add"],term),
                    "main_effect_partial_r2":effect_size})
            for i,((aa,bb),term) in enumerate(zip(combinations(triple,2),fit["pair_terms"])):
                remaining=[t for j,t in enumerate(fit["pair_terms"]) if j!=i]
                reduced=ols("response ~ "+" + ".join(fit["terms"]+remaining),fit["model_data"]).fit()
                rss_reduced=float(np.sum(reduced.resid**2))
                effect_size=max(0.,(rss_reduced-rss_second)/rss_reduced) if rss_reduced>0 else np.nan
                second_records.append({"hrv_variable":col,"triplet":name,"factor_a":aa,"factor_b":bb,
                    "model":"all_pair_interactions_HC3","term":term,"p_hc3":wald_p(fit["second"],term),
                    "second_order_partial_r2":effect_size})
        categorical_records.append(record)

categorical=pd.DataFrame(categorical_records)
main_effects=pd.DataFrame(main_records)
second_effects=pd.DataFrame(second_records)

def attach_bh(table,p_col,q_col):
    table[q_col]=np.nan
    if len(table)==0:return table
    valid=np.isfinite(table[p_col].to_numpy(float))
    if valid.any():
        table.loc[valid,q_col]=multipletests(table.loc[valid,p_col],method="fdr_bh")[1]
    return table

categorical=attach_bh(categorical,"three_way_p_hc3","three_way_q_bh_28")
main_effects=attach_bh(main_effects,"p_hc3","q_bh_all_84_main_tests")
second_effects=attach_bh(second_effects,"p_hc3","q_bh_all_84_pair_tests")
categorical.to_csv(TABLE_DIR / "categorical_three_way_models.csv",index=False)
main_effects.to_csv(TABLE_DIR / "adjusted_main_effects.csv",index=False)
second_effects.to_csv(TABLE_DIR / "two_way_terms_adjusted_for_third.csv",index=False)
print("Categorical models:",len(categorical),"estimated:",categorical.model_status.eq("ok").sum())
print("HC3 main-effect tests:",len(main_effects),"two-way tests:",len(second_effects))
display(categorical.round(4))

Categorical models: 28 estimated: 28
HC3 main-effect tests: 84 two-way tests: 84


,hrv_variable,triplet,factor_a,factor_b,factor_c,n,min_analysis_cell_n,model_scale,model_status,low_support_warning,...,three_way_partial_r2,full_model_r2,full_model_adj_r2,full_model_aic,condition_number,max_leverage,max_cooks_distance,parameters,rank,three_way_q_bh_28
0,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,476,13,identity,ok,False,...,0.0021,0.0371,0.0143,5948.0027,3.7802,0.0769,0.0231,12,12,0.9099
1,t2m_pre_mean_rr,age_x_sex_x_bmi,age_group,sex_category,bmi_category,476,11,identity,ok,False,...,0.0075,0.0344,0.0115,5949.3564,3.8927,0.0909,0.0294,12,12,0.7680
2,t2m_pre_mean_rr,age_x_bp_x_bmi,age_group,bp_category,bmi_category,476,12,identity,ok,False,...,0.0059,0.0470,0.0116,5955.1150,4.3201,0.0833,0.0254,18,18,0.9099
3,t2m_pre_mean_rr,sex_x_bp_x_bmi,sex_category,bp_category,bmi_category,476,9,identity,ok,True,...,0.0055,0.0305,-0.0055,5963.2614,5.5744,0.1111,0.0291,18,18,0.9099
4,t2m_pre_mean_hr,age_x_sex_x_bp,age_group,sex_category,bp_category,476,13,identity,ok,False,...,0.0014,0.0336,0.0107,3643.3866,3.7802,0.0769,0.0386,12,12,0.9099
5,t2m_pre_mean_hr,age_x_sex_x_bmi,age_group,sex_category,bmi_category,476,11,identity,ok,False,...,0.0085,0.0320,0.0090,3644.2131,3.8927,0.0909,0.0454,12,12,0.7680
6,t2m_pre_mean_hr,age_x_bp_x_bmi,age_group,bp_category,bmi_category,476,12,identity,ok,False,...,0.0071,0.0468,0.0115,3648.8424,4.3201,0.0833,0.0337,18,18,0.9099
7,t2m_pre_mean_hr,sex_x_bp_x_bmi,sex_category,bp_category,bmi_category,476,9,identity,ok,True,...,0.0048,0.0286,-0.0074,3657.8462,5.5744,0.1111,0.0542,18,18,0.9099
8,t2m_pre_sdnn,age_x_sex_x_bp,age_group,sex_category,bp_category,476,13,log10,ok,False,...,0.0042,0.1452,0.1249,592.2569,3.7802,0.0769,0.0305,12,12,0.9099
9,t2m_pre_sdnn,age_x_sex_x_bmi,age_group,sex_category,bmi_category,476,11,log10,ok,False,...,0.0048,0.1431,0.1228,593.3995,3.8927,0.0909,0.0367,12,12,0.9099


## 4. Conditional nonparametric comparisons and model-based contrasts

Every factor is a possible **focal factor**. Its levels are compared **within each joint combination of the other two factors**, subject to both compared cells having at least 10 participants. Comparisons are independent cross-sectional groups. Cliff's delta direction is **first level minus second level**; median differences have the same direction. Percentile bootstrap intervals are exploratory uncertainty estimates, and Dunn's procedure is not used because these comparisons are already planned pairwise/conditioned tests.

Model-based contrasts use the three-way HC3 fit; on log10 outcome scales, the contrast is reported both on the log scale and as a **ratio of geometric means**. These contrasts are derived from the full model (not duplicated Mann–Whitney p-values). FDR is computed within each HRV×triplet family and across all pooled contrasts separately.

In [5]:
def boot_pair(x,y,seed,reps=BOOTSTRAP_REPLICATES):
    rng=np.random.default_rng(seed)
    n,m=len(x),len(y)
    bx=x[rng.integers(0,n,size=(reps,n))]
    by=y[rng.integers(0,m,size=(reps,m))]
    ranks=stats.rankdata(np.concatenate([bx,by],axis=1),method="average",axis=1)
    u=ranks[:,:n].sum(axis=1)-n*(n+1)/2
    delta=2*u/(n*m)-1
    differences=np.median(bx,axis=1)-np.median(by,axis=1)
    return np.quantile(delta,[.025,.975]),np.quantile(differences,[.025,.975])

def model_simple_contrast(fit, three_levels_1, three_levels_2):
    model=fit["full"]
    design_info=fit["full_design_info"]
    keys=list(three_levels_1)
    new=pd.DataFrame([three_levels_1,three_levels_2])
    for f in keys:
        original_factor=next(ff for ff in FACTORS if ANALYSIS_COL[ff]==f)
        new[f]=pd.Categorical(new[f],categories=ANALYSIS_LEVELS[original_factor],ordered=True)
    X=build_design_matrices([design_info],new)[0]
    linear=np.asarray(X[0]-X[1],dtype=float)
    test=model.t_test(linear)
    ci=np.asarray(test.conf_int(alpha=.05)).reshape(-1)
    effect=float(np.asarray(test.effect).item())
    return effect,float(ci[0]),float(ci[1]),float(np.asarray(test.pvalue).item())

# Pre-flight: contrast design must agree with differences in model predictions.
# The check uses the fitted training specification, not a reconstructed coding guess.
for (sample_col, sample_triplet), sample_fit in list(cat_fits.items())[:2]:
    triplet = next(t for t in TRIPLETS if TRIPLET_NAMES[t] == sample_triplet)
    pair_a, pair_b, pair_c = [ANALYSIS_COL[f] for f in triplet]
    level_set = [ANALYSIS_LEVELS[f] for f in triplet]
    point_a = {pair_a: level_set[0][0], pair_b: level_set[1][0], pair_c: level_set[2][0]}
    point_b = {**point_a, pair_a: level_set[0][1]}
    estimate, *_ = model_simple_contrast(sample_fit, point_a, point_b)
    candidate = pd.DataFrame([point_a, point_b])
    for f in triplet:
        key = ANALYSIS_COL[f]
        candidate[key] = pd.Categorical(candidate[key], categories=ANALYSIS_LEVELS[f], ordered=True)
    predicted = prediction_summary(sample_fit["full"], sample_fit["full_design_info"], candidate)
    expected = float(predicted["mean"].iloc[0] - predicted["mean"].iloc[1])
    if not np.isclose(estimate, expected, atol=1e-10, rtol=1e-10):
        raise AssertionError("Model-based contrast and prediction difference disagree")
print("Patsy design and HC3 contrast pre-flight: passed")

contrast_rows=[]
model_contrast_rows=[]
idx=0
for col in HRV:
    for triple in TRIPLETS:
        name=TRIPLET_NAMES[triple]
        fit=cat_fits.get((col,name))
        for focal in triple:
            remaining=[f for f in triple if f != focal]
            r1,r2=remaining
            focal_key=ANALYSIS_COL[focal]
            for lev1 in ANALYSIS_LEVELS[r1]:
                for lev2 in ANALYSIS_LEVELS[r2]:
                    stratum={ANALYSIS_COL[r1]:lev1,ANALYSIS_COL[r2]:lev2}
                    ds=df.loc[(df[ANALYSIS_COL[r1]]==lev1)&(df[ANALYSIS_COL[r2]]==lev2)]
                    for first,second in combinations(ANALYSIS_LEVELS[focal],2):
                        x=ds.loc[ds[focal_key]==first,col].to_numpy(float)
                        y=ds.loc[ds[focal_key]==second,col].to_numpy(float)
                        record={"hrv_variable":col,"triplet":name,"focal_factor":focal,
                            "conditioning_factor_1":r1,"conditioning_factor_2":r2,
                            "conditioning_level_1":lev1,"conditioning_level_2":lev2,
                            "level_1":first,"level_2":second,"n_1":len(x),"n_2":len(y),
                            "min_contrast_n":MIN_CONTRAST_N,"estimable":min(len(x),len(y))>=MIN_CONTRAST_N}
                        if record["estimable"]:
                            u=stats.mannwhitneyu(x,y,alternative="two-sided",method="asymptotic")
                            cliffs=2*u.statistic/(len(x)*len(y))-1
                            cliffs_ci,median_ci=boot_pair(x,y,SEED+500000+idx)
                            ks=stats.ks_2samp(x,y)
                            bf=stats.levene(x,y,center="median")
                            record.update({"median_1":float(np.median(x)),"median_2":float(np.median(y)),
                                "median_difference_1_minus_2":float(np.median(x)-np.median(y)),
                                "median_diff_ci95_low":float(median_ci[0]),"median_diff_ci95_high":float(median_ci[1]),
                                "cliffs_delta":float(cliffs),"cliffs_ci95_low":float(cliffs_ci[0]),
                                "cliffs_ci95_high":float(cliffs_ci[1]),"mw_p":float(u.pvalue),
                                "ks_statistic":float(ks.statistic),"ks_p":float(ks.pvalue),
                                "wasserstein_distance":float(stats.wasserstein_distance(x,y)),
                                "brown_forsythe_p":float(bf.pvalue)})
                        contrast_rows.append(record)
                        if fit is not None:
                            one={**stratum,focal_key:first}
                            two={**stratum,focal_key:second}
                            est,low,high,p=model_simple_contrast(fit,one,two)
                            model_contrast_rows.append({**{k:v for k,v in record.items() if k not in ["estimable"]},
                                "small_cell_warning":min(len(x),len(y))<MIN_CONTRAST_N,
                                "model_scale":SCALES[col],"model_contrast":est,
                                "contrast_ci95_low":low,"contrast_ci95_high":high,"p_hc3":p,
                                "geometric_mean_ratio":10**est if SCALES[col]=="log10" else np.nan,
                                "geometric_ratio_ci95_low":10**low if SCALES[col]=="log10" else np.nan,
                                "geometric_ratio_ci95_high":10**high if SCALES[col]=="log10" else np.nan})
                        idx+=1

conditional=pd.DataFrame(contrast_rows)
conditional_models=pd.DataFrame(model_contrast_rows)
for tab,p_col,prefix in [(conditional,"mw_p","mw"),(conditional,"ks_p","ks"),(conditional,"brown_forsythe_p","bf"),
                         (conditional_models,"p_hc3","hc3")]:
    tab[f"{prefix}_q_global"] = np.nan
    if len(tab) == 0:continue
    tab[f"{prefix}_q_within_hrv_triplet"] = np.nan
    for key,group in tab.groupby(["hrv_variable","triplet"]):
        valid=group[p_col].notna()
        if valid.any():
            inds=group.loc[valid].index
            tab.loc[inds,f"{prefix}_q_within_hrv_triplet"]=multipletests(tab.loc[inds,p_col],method="fdr_bh")[1]
    valid=tab[p_col].notna()
    if valid.any():
        tab.loc[valid,f"{prefix}_q_global"]=multipletests(tab.loc[valid,p_col],method="fdr_bh")[1]

conditional.to_csv(TABLE_DIR / "conditional_nonparametric_comparisons.csv",index=False)
conditional_models.to_csv(TABLE_DIR / "conditional_hc3_model_contrasts.csv",index=False)
print(f"Planned comparisons: {len(conditional)} | nonparametric eligible: {conditional.estimable.sum()}")
print(f"Model-based HC3 conditional contrasts: {len(conditional_models)}")
display(conditional.loc[conditional.estimable].head(10).round(3))

Patsy design and HC3 contrast pre-flight: passed
Planned comparisons: 966 | nonparametric eligible: 903
Model-based HC3 conditional contrasts: 966


,hrv_variable,triplet,focal_factor,conditioning_factor_1,conditioning_factor_2,conditioning_level_1,conditioning_level_2,level_1,level_2,n_1,...,ks_statistic,ks_p,wasserstein_distance,brown_forsythe_p,mw_q_global,mw_q_within_hrv_triplet,ks_q_global,ks_q_within_hrv_triplet,bf_q_global,bf_q_within_hrv_triplet
0,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,Female,Optimal/Normal,60-69,70+,72,...,0.120,0.545,28.334,0.332,0.916,0.705,0.990,0.775,0.851,0.948
1,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,Female,High-normal,60-69,70+,34,...,0.238,0.217,54.787,0.158,0.683,0.456,0.885,0.652,0.851,0.948
2,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,Female,Stage 1-3,60-69,70+,44,...,0.210,0.208,36.608,0.643,0.864,0.549,0.885,0.652,0.905,0.948
3,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,Male,Optimal/Normal,60-69,70+,18,...,0.215,0.529,40.110,0.286,0.919,0.705,0.990,0.775,0.851,0.948
4,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,Male,High-normal,60-69,70+,13,...,0.437,0.061,106.129,0.227,0.540,0.315,0.808,0.607,0.851,0.948
5,t2m_pre_mean_rr,age_x_sex_x_bp,age_group,sex_category,bp_category,Male,Stage 1-3,60-69,70+,17,...,0.199,0.657,44.392,0.484,0.950,0.827,0.995,0.775,0.864,0.948
6,t2m_pre_mean_rr,age_x_sex_x_bp,sex_category,age_group,bp_category,60-69,Optimal/Normal,Female,Male,72,...,0.208,0.524,27.500,0.611,0.916,0.705,0.990,0.775,0.904,0.948
7,t2m_pre_mean_rr,age_x_sex_x_bp,sex_category,age_group,bp_category,60-69,High-normal,Female,Male,34,...,0.351,0.154,39.552,0.520,0.888,0.632,0.882,0.652,0.872,0.948
8,t2m_pre_mean_rr,age_x_sex_x_bp,sex_category,age_group,bp_category,60-69,Stage 1-3,Female,Male,44,...,0.247,0.370,38.361,0.226,0.945,0.827,0.989,0.775,0.851,0.948
9,t2m_pre_mean_rr,age_x_sex_x_bp,sex_category,age_group,bp_category,70+,Optimal/Normal,Female,Male,94,...,0.162,0.431,30.788,0.839,0.882,0.549,0.990,0.775,0.955,0.948


## 5. Continuous and mixed three-factor models

Seven specifications mirror the physical variables used in Notebook 07: `age × sex × SBP/DBP`, `age × sex × BMI`, `age × BMI × SBP/DBP`, and `sex × BMI × SBP/DBP`. Continuous predictors are mean-centered and scaled for interpretable coefficient magnitudes; sex uses sum contrasts.

Third-order tests use HC3-robust Wald contrasts. Second-order tests are evaluated in hierarchical models with **all three pairwise interactions but without the three-way term**; first-order effects are evaluated in additive three-factor models. Separate BH families are used for 49 third-order, 147 second-order and 147 main-effect tests.

In [6]:
CONTINUOUS_DEFS = {
    "age":("age_10","Age",10.0,"years"),
    "imc":("bmi_5","BMI",5.0,"kg/m²"),
    "bp_systolic":("sbp_10","SBP",10.0,"mmHg"),
    "bp_diastolic":("dbp_10","DBP",10.0,"mmHg"),
}
for raw,(column,label,step,_unit) in CONTINUOUS_DEFS.items():
    df[column]=(df[raw]-df[raw].mean())/step

SPECS=[
    ("age_sex_sbp",("age","sex_category","bp_systolic"),("age_group","sex_category","bp_category")),
    ("age_sex_dbp",("age","sex_category","bp_diastolic"),("age_group","sex_category","bp_category")),
    ("age_sex_bmi",("age","sex_category","imc"),("age_group","sex_category","bmi_category")),
    ("age_bmi_sbp",("age","imc","bp_systolic"),("age_group","bmi_category","bp_category")),
    ("age_bmi_dbp",("age","imc","bp_diastolic"),("age_group","bmi_category","bp_category")),
    ("sex_bmi_sbp",("sex_category","imc","bp_systolic"),("sex_category","bmi_category","bp_category")),
    ("sex_bmi_dbp",("sex_category","imc","bp_diastolic"),("sex_category","bmi_category","bp_category")),
]

def continuous_term(raw):
    return f"C({raw}, Sum)" if raw=="sex_category" else CONTINUOUS_DEFS[raw][0]

continuous_records=[]
continuous_main_records=[]
continuous_second_records=[]
continuous_fits={}
continuous_design_infos={}
for col in HRV:
    for spec,raws,factors in SPECS:
        terms=[continuous_term(raw) for raw in raws]
        d=df.copy()
        d["response"] = np.log10(d[col]) if SCALES[col]=="log10" else d[col].astype(float)
        pair_terms=[terms[0]+":"+terms[1],terms[0]+":"+terms[2],terms[1]+":"+terms[2]]
        formula_base="response ~ "+" + ".join(terms)
        try:
            add=ols(formula_base,data=d).fit(cov_type="HC3")
            second=ols(formula_base+" + "+" + ".join(pair_terms),data=d).fit(cov_type="HC3")
            full=ols("response ~ "+" * ".join(terms),data=d).fit(cov_type="HC3")
            status="ok" if all(m.model.rank==m.model.exog.shape[1] for m in (add,second,full)) else "rank_deficient"
        except (ValueError,np.linalg.LinAlgError):
            status="fit_error"
        result={"hrv_variable":col,"specification":spec,"factor_triplet":TRIPLET_NAMES[tuple(sorted(factors,key=list(FACTORS).index))],
            "raw_factors":" | ".join(raws),"outcome_scale":SCALES[col],"n":len(d),"model_status":status}
        if status=="ok":
            rss2=float(np.sum(second.resid**2));rss3=float(np.sum(full.resid**2))
            term_three=":".join(terms)
            result.update({"three_way_p_hc3":wald_p(full,term_three),
                "three_way_partial_r2":max(0.,(rss2-rss3)/rss2) if rss2>0 else np.nan,
                "full_model_r2":float(full.rsquared),"condition_number":float(full.condition_number),
                "max_leverage":float(full.get_influence().hat_matrix_diag.max())})
            continuous_fits[(col,spec)]=full
            continuous_design_infos[(col,spec)]=fitted_design_info(full,d)
            for raw,term in zip(raws,terms):
                continuous_main_records.append({"hrv_variable":col,"specification":spec,"factor":raw,
                    "term":term,"p_hc3":wald_p(add,term)})
            for (fa,fb),term in zip(combinations(raws,2),pair_terms):
                continuous_second_records.append({"hrv_variable":col,"specification":spec,
                    "factor_a":fa,"factor_b":fb,"term":term,"p_hc3":wald_p(second,term)})
        continuous_records.append(result)
continuous=pd.DataFrame(continuous_records)
continuous_main=pd.DataFrame(continuous_main_records)
continuous_second=pd.DataFrame(continuous_second_records)
continuous=attach_bh(continuous,"three_way_p_hc3","three_way_q_bh_49")
continuous_main=attach_bh(continuous_main,"p_hc3","q_bh_147_main")
continuous_second=attach_bh(continuous_second,"p_hc3","q_bh_147_pair")
continuous.to_csv(TABLE_DIR / "continuous_three_way_models.csv",index=False)
continuous_main.to_csv(TABLE_DIR / "continuous_adjusted_main_effects.csv",index=False)
continuous_second.to_csv(TABLE_DIR / "continuous_two_way_terms.csv",index=False)
print(f"Continuous/mixed models: {len(continuous)} | estimated: {(continuous.model_status=='ok').sum()}")
display(continuous.round(4))

Continuous/mixed models: 49 | estimated: 49


,hrv_variable,specification,factor_triplet,raw_factors,outcome_scale,n,model_status,three_way_p_hc3,three_way_partial_r2,full_model_r2,condition_number,max_leverage,three_way_q_bh_49
0,t2m_pre_mean_rr,age_sex_sbp,age_x_sex_x_bp,age | sex_category | bp_systolic,identity,476,ok,0.5087,0.0011,0.0204,4.6759,0.3652,0.7579
1,t2m_pre_mean_rr,age_sex_dbp,age_x_sex_x_bp,age | sex_category | bp_diastolic,identity,476,ok,0.7823,0.0003,0.0601,2.8851,0.1800,0.8914
2,t2m_pre_mean_rr,age_sex_bmi,age_x_sex_x_bmi,age | sex_category | imc,identity,476,ok,0.4642,0.0016,0.0190,3.5880,0.1807,0.7579
3,t2m_pre_mean_rr,age_bmi_sbp,age_x_bp_x_bmi,age | imc | bp_systolic,identity,476,ok,0.4285,0.0016,0.0358,3.3949,0.1805,0.7579
4,t2m_pre_mean_rr,age_bmi_dbp,age_x_bp_x_bmi,age | imc | bp_diastolic,identity,476,ok,0.9117,0.0000,0.0605,2.5318,0.2587,0.9712
5,t2m_pre_mean_rr,sex_bmi_sbp,sex_x_bp_x_bmi,sex_category | imc | bp_systolic,identity,476,ok,0.2967,0.0035,0.0224,3.2570,0.2031,0.7579
6,t2m_pre_mean_rr,sex_bmi_dbp,sex_x_bp_x_bmi,sex_category | imc | bp_diastolic,identity,476,ok,0.9746,0.0000,0.0542,2.1636,0.1886,0.9765
7,t2m_pre_mean_hr,age_sex_sbp,age_x_sex_x_bp,age | sex_category | bp_systolic,identity,476,ok,0.5672,0.0008,0.0180,4.6759,0.3652,0.8009
8,t2m_pre_mean_hr,age_sex_dbp,age_x_sex_x_bp,age | sex_category | bp_diastolic,identity,476,ok,0.9765,0.0000,0.0592,2.8851,0.1800,0.9765
9,t2m_pre_mean_hr,age_sex_bmi,age_x_sex_x_bmi,age | sex_category | imc,identity,476,ok,0.3686,0.0027,0.0180,3.5880,0.1807,0.7579


## 6. Sensitivity, influential observations and original age feasibility

- **Trimming:** refit after excluding each response's outer 1% *only in that sensitivity fit*. The main dataset is unchanged.
- **Scale:** refit the same categorical model on the other eligible HRV scale.
- **Age grouping:** check whether the uncollapsed age 60–69/70–79/80+ scheme reaches minimum model-cell support (no imputation or forced model fit when unsupported).
- **Diagnostics:** document full-model leverage, condition number and Cook's distance. These are warnings, not automated exclusions.

In [7]:
sensitivity_rows=[]
for col in HRV:
    primary_scale=SCALES[col]
    alternative="identity" if primary_scale=="log10" else "log10"
    low,high=df[col].quantile([.01,.99])
    trimmed=df.loc[df[col].between(low,high)]
    for triple in TRIPLETS:
        name=TRIPLET_NAMES[triple]
        primary=categorical.loc[(categorical.hrv_variable==col)&(categorical.triplet==name)].iloc[0]
        for label,subset,scale in [
            ("trim_outer_one_percent_each_tail",trimmed,primary_scale),
            ("alternative_outcome_scale",df,alternative),
        ]:
            fit,status,min_n=fit_categorical(col,triple,subset,scale)
            rec={"hrv_variable":col,"triplet":name,"sensitivity":label,
                "n_primary":len(df),"n_sensitivity":len(subset),"scale_primary":primary_scale,
                "scale_sensitivity":scale,"min_analysis_cell_n":min_n,"status":status,
                "primary_three_way_p":primary.get("three_way_p_hc3",np.nan),
                "primary_partial_r2":primary.get("three_way_partial_r2",np.nan),
                "sensitivity_three_way_p":wald_p(fit["full"],fit["triple_term"]) if fit else np.nan,
                "sensitivity_partial_r2":fit["partial_r2"] if fit else np.nan,
                "trim_q01":low if label.startswith("trim") else np.nan,
                "trim_q99":high if label.startswith("trim") else np.nan}
            sensitivity_rows.append(rec)
        # Explicit feasibility of original age strata with the same BP/BMI analysis groupings.
        if "age_group" in triple:
            old_age=df["age_analysis"].copy()
            df["age_analysis"]=pd.Categorical(df["age_group"],categories=FACTORS["age_group"],ordered=True)
            original_grid=df.groupby([ANALYSIS_COL[f] for f in triple],observed=True).size().reindex(
                pd.MultiIndex.from_product([FACTORS["age_group"] if f=="age_group" else ANALYSIS_LEVELS[f] for f in triple]),fill_value=0)
            df["age_analysis"]=old_age
            sensitivity_rows.append({"hrv_variable":col,"triplet":name,
                "sensitivity":"original_age_three_levels_feasibility_only", "n_primary":len(df),
                "n_sensitivity":len(df),"scale_primary":primary_scale,"scale_sensitivity":primary_scale,
                "min_analysis_cell_n":int(original_grid.min()),
                "status":"meets_min_cell_threshold" if int(original_grid.min())>=MIN_MODEL_CELL_N else "insufficient_cell_support",
                "primary_three_way_p":primary.get("three_way_p_hc3",np.nan),
                "primary_partial_r2":primary.get("three_way_partial_r2",np.nan)})
sensitivity=pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(TABLE_DIR / "trifactorial_sensitivity.csv",index=False)
print("Sensitivity/feasibility records:",len(sensitivity))
display(sensitivity.groupby(["sensitivity","status"]).size().rename("n").to_frame())

Sensitivity/feasibility records: 77


,,n
sensitivity,status,
alternative_outcome_scale,ok,28
original_age_three_levels_feasibility_only,insufficient_cell_support,21
trim_outer_one_percent_each_tail,ok,28


## 7. Per-HRV/per-triplet figures — original category distributions and conditioned ECDFs

Figures remain organized by metric and triplet. Colors identify the **second factor** in distributions; panels identify the third factor. Original levels (including small/empty cells) are kept visible; the sample size of each facet is stated. Log axes are **display-only** and do not alter the dataset.

In [8]:
def use_log_axis(col):
    sample=df[col].to_numpy(float)
    return bool(sample.min()>0 and np.quantile(sample,.99)/max(np.quantile(sample,.01),1e-12)>30)

def savefig(fig,path,dpi=145):
    fig.savefig(path,dpi=dpi,bbox_inches="tight")
    plt.close(fig)

for col,(slug,label,unit) in HRV.items():
    for triple in TRIPLETS:
        a,b,c=triple
        folder=FIGURE_DIR/slug/TRIPLET_NAMES[triple]
        levels_c=FACTORS[c]
        ncols=min(3,len(levels_c));nrows=int(np.ceil(len(levels_c)/ncols))
        log_y=use_log_axis(col)
        fig,axes=plt.subplots(nrows,ncols,figsize=(5.9*ncols,4.9*nrows),squeeze=False)
        for ax,lc in zip(axes.flat,levels_c):
            subset=df.loc[df[c]==lc]
            if len(subset):
                sns.boxplot(data=subset,x=a,y=col,hue=b,order=FACTORS[a],hue_order=FACTORS[b],
                    palette=PALETTES[b],showfliers=False,ax=ax,linewidth=1.1)
                sns.stripplot(data=subset,x=a,y=col,order=FACTORS[a],color="#263241",
                    alpha=.19,size=2.2,jitter=.24,ax=ax)
            if log_y:ax.set_yscale("log")
            ax.set(title=f"{FACTOR_LABELS[c]} = {lc} | n={len(subset)}",xlabel=FACTOR_LABELS[a],ylabel=f"{label} ({unit})")
            ax.tick_params(axis="x",rotation=20)
            if ax.legend_:
                ax.legend_.set_title(FACTOR_LABELS[b])
                for legend_text in ax.legend_.get_texts():
                    legend_text.set_fontsize(8)
            sns.despine(ax=ax)
        for ax in list(axes.flat)[len(levels_c):]:ax.axis("off")
        fig.suptitle(f"{label} | original-category distributions | {TRIPLET_NAMES[triple]}",fontsize=17,fontweight="semibold")
        fig.tight_layout(rect=(0,0,1,.965),h_pad=2.1,w_pad=2)
        savefig(fig,folder/"01_original_distributions.png")

        # ECDF panels: within the third factor, compare second-factor levels.
        fig,axes=plt.subplots(nrows,ncols,figsize=(5.9*ncols,4.7*nrows),squeeze=False)
        for ax,lc in zip(axes.flat,levels_c):
            subset=df.loc[df[c]==lc]
            if len(subset):
                sns.ecdfplot(data=subset,x=col,hue=b,hue_order=FACTORS[b],palette=PALETTES[b],linewidth=1.8,ax=ax)
            if log_y:ax.set_xscale("log")
            ax.set(title=f"{FACTOR_LABELS[c]} = {lc} | n={len(subset)}",xlabel=f"{label} ({unit})",ylabel="Cumulative fraction")
            if ax.legend_:ax.legend_.set_title(FACTOR_LABELS[b])
        for ax in list(axes.flat)[len(levels_c):]:ax.axis("off")
        fig.suptitle(f"{label} | ECDFs conditioned on {FACTOR_LABELS[c]}",fontsize=17,fontweight="semibold")
        fig.tight_layout(rect=(0,0,1,.96),h_pad=2)
        savefig(fig,folder/"02_original_ecdf_by_third_factor.png")
print("Saved 56 original-category distribution figures")

Saved 56 original-category distribution figures


## 8. Per-HRV/per-triplet model figures — conditioned medians, predicted interactions, effect contrasts, model diagnostics

Facet count/median heatmaps use **analysis categories** (original-category cell counts are already available). Predicted interaction plots visualize a full **three-way** HC3 model, with the third factor in separate panels; backtransformed log-scale estimates are geometric means. Nonparametric forests prioritize the largest eligible Cliff's delta effects, **not statistical significance**, to avoid ranking by p-value alone. Diagnostic plots display residuals, QQ and leverage/Cook's distance.

In [9]:
for col,(slug,label,unit) in HRV.items():
    for triple in TRIPLETS:
        name=TRIPLET_NAMES[triple]
        folder=FIGURE_DIR/slug/name
        a,b,c=triple
        aa,bb,cc=[ANALYSIS_COL[f] for f in triple]
        levels=ANALYSIS_LEVELS[c]
        fig,axes=plt.subplots(1,len(levels),figsize=(5.0*len(levels),4.7),squeeze=False)
        for ax,lv_c in zip(axes.flat,levels):
            subset=df.loc[df[cc]==lv_c]
            group=subset.groupby([aa,bb],observed=True)[col].agg(["median","size"]).reset_index()
            median=group.pivot(index=aa,columns=bb,values="median").reindex(index=ANALYSIS_LEVELS[a],columns=ANALYSIS_LEVELS[b])
            count=group.pivot(index=aa,columns=bb,values="size").reindex(index=ANALYSIS_LEVELS[a],columns=ANALYSIS_LEVELS[b]).fillna(0)
            annotations=np.array([[f"{median.iloc[i,j]:.2g}\nn={int(count.iloc[i,j])}" if count.iloc[i,j]>0 else "—"
                for j in range(median.shape[1])] for i in range(median.shape[0])],dtype=str)
            sns.heatmap(median,ax=ax,cmap="YlGnBu",annot=annotations,fmt="",linewidths=.6,
                linecolor="white",cbar_kws={"label":f"Median {label} ({unit})"},annot_kws={"size":9})
            ax.set(title=f"{FACTOR_LABELS[c]}={lv_c}",xlabel=FACTOR_LABELS[b],ylabel=FACTOR_LABELS[a])
            ax.tick_params(axis="x",rotation=25)
        fig.suptitle(f"{label} | conditioned medians and cell support",fontsize=17,fontweight="semibold")
        fig.tight_layout(rect=(0,0,1,.94),w_pad=1)
        savefig(fig,folder/"03_conditioned_medians.png")

        fit=cat_fits.get((col,name))
        if fit is not None:
            model=fit["full"]
            fig,axes=plt.subplots(1,len(levels),figsize=(5.8*len(levels),4.9),squeeze=False)
            for ax,lc in zip(axes.flat,levels):
                for lb in ANALYSIS_LEVELS[b]:
                    grid=pd.DataFrame({aa:ANALYSIS_LEVELS[a],bb:[lb]*len(ANALYSIS_LEVELS[a]),cc:[lc]*len(ANALYSIS_LEVELS[a])})
                    for f in triple:
                        key=ANALYSIS_COL[f]
                        grid[key]=pd.Categorical(grid[key],categories=ANALYSIS_LEVELS[f],ordered=True)
                    means=prediction_summary(model, fit["full_design_info"], grid, alpha=.05)
                    point=means["mean"].to_numpy();lower=means["mean_ci_lower"].to_numpy();upper=means["mean_ci_upper"].to_numpy()
                    if SCALES[col]=="log10":point,lower,upper=10**point,10**lower,10**upper
                    xx=np.arange(len(ANALYSIS_LEVELS[a]))
                    ax.errorbar(xx,point,yerr=[point-lower,upper-point],marker="o",markersize=5,lw=2.0,
                        capsize=3.5,color=ANALYSIS_PALETTES[b][lb],label=lb)
                ax.set_xticks(np.arange(len(ANALYSIS_LEVELS[a])),ANALYSIS_LEVELS[a])
                ax.set(title=f"{FACTOR_LABELS[c]} = {lc}",xlabel=FACTOR_LABELS[a],
                    ylabel=f"{'Geometric' if SCALES[col]=='log10' else 'Arithmetic'} mean {label} ({unit})")
                if use_log_axis(col):ax.set_yscale("log")
                ax.legend(title=FACTOR_LABELS[b],fontsize=8)
                sns.despine(ax=ax)
            current=categorical.loc[(categorical.hrv_variable==col)&(categorical.triplet==name)].iloc[0]
            fig.suptitle(f"{label} | three-way interaction by {FACTOR_LABELS[c]}",fontsize=18,fontweight="semibold")
            fig.text(.5,.02,f"HC3 three-way p={current.three_way_p_hc3:.3g}, BH q={current.three_way_q_bh_28:.3g} | partial R²={current.three_way_partial_r2:.3f}",ha="center",fontsize=10)
            fig.tight_layout(rect=(0,.06,1,.94),w_pad=1.6)
            savefig(fig,folder/"04_hc3_three_way_interaction.png")

            # Largest eligible nonparametric conditional effects, maintaining actual direction.
            sub=conditional.loc[(conditional.hrv_variable==col)&(conditional.triplet==name)&conditional.estimable].copy()
            if len(sub):
                sub=sub.sort_values("cliffs_delta",key=lambda x:x.abs(),ascending=False).head(24).iloc[::-1]
                labels=(sub.level_1+" vs "+sub.level_2+" | "+sub.conditioning_level_1+" & "+sub.conditioning_level_2)
                fig,ax=plt.subplots(figsize=(12.8,max(5.5,0.37*len(sub)+2)))
                ys=np.arange(len(sub))
                colors=["#ba4b55" if q<FDR_ALPHA else "#3e779b" for q in sub.mw_q_within_hrv_triplet]
                for i,r in enumerate(sub.itertuples()):
                    ax.plot([r.cliffs_ci95_low,r.cliffs_ci95_high],[ys[i],ys[i]],lw=2.0,color=colors[i])
                    ax.plot(r.cliffs_delta,ys[i],marker="o",ms=5,color=colors[i])
                ax.set_yticks(ys,labels,fontsize=8)
                ax.axvline(0,color="black",lw=1,ls="--");ax.set_xlim(-1.05,1.05)
                ax.set(xlabel="Cliff's delta (95% bootstrap CI)",ylabel="",
                    title=f"{label}: largest eligible conditional contrasts | {name}")
                ax.grid(axis="y",visible=False);fig.tight_layout()
                savefig(fig,folder/"05_conditional_cliffs_forest.png")
            else:
                # Explicit note; no fabricated image for unestimable contrasts.
                print(f"No conditional nonparametric comparisons estimated for {col}/{name}")

            # Model diagnostics (fitted on the model-response scale).
            influence=model.get_influence()
            residual=np.asarray(model.resid);fitted=np.asarray(model.fittedvalues)
            leverage=np.asarray(influence.hat_matrix_diag)
            cooks=np.asarray(influence.cooks_distance[0])
            fig,axs=plt.subplots(1,3,figsize=(16,4.9))
            sns.scatterplot(x=fitted,y=residual,ax=axs[0],s=17,alpha=.42,color="#2f7590")
            axs[0].axhline(0,color="black",ls="--",lw=1)
            axs[0].set(xlabel="Fitted response (model scale)",ylabel="Residual",title="Residuals vs fitted")
            stats.probplot(residual,dist="norm",plot=axs[1]);axs[1].set_title("Residual Q-Q")
            sns.scatterplot(x=leverage,y=cooks,ax=axs[2],s=17,alpha=.45,color="#9e5d53")
            axs[2].axhline(4/len(df),color="black",ls="--",lw=1,label="4/n")
            axs[2].set(title="Leverage and influence",xlabel="Leverage",ylabel="Cook's distance")
            axs[2].legend(fontsize=8)
            fig.suptitle(f"{label} | model diagnostics | {name}",fontsize=17,fontweight="semibold")
            fig.tight_layout(rect=(0,0,1,.94))
            savefig(fig,folder/"06_model_diagnostics.png")
print("Completed core triplet figures")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Completed core triplet figures


## 9. Continuous/mixed three-way interaction figures

For each of seven continuous/mixed specifications and each HRV metric, show the response against one continuous factor. Curves are evaluated at low/high (25th/75th percentile) values of a conditioning continuous factor and faceted by the remaining factor (sex or a further continuous factor at its 25th/75th percentiles). Curves and confidence intervals come from the **same HC3 interaction model**; all input measurements remain unchanged. Sample-size and support cautions remain in the statistical tables.

In [10]:
for col,(slug,label,unit) in HRV.items():
    for spec,raws,factors in SPECS:
        fit=continuous_fits.get((col,spec))
        if fit is None:continue
        triple=tuple(sorted(factors,key=list(FACTORS).index))
        folder=FIGURE_DIR/slug/TRIPLET_NAMES[triple]
        # x = first non-sex continuous predictor; z = second non-sex predictor;
        # third = categorical sex if present, else the third continuous predictor.
        continuous_raws=[f for f in raws if f!="sex_category"]
        x_raw,z_raw=continuous_raws[:2]
        panel_raw="sex_category" if "sex_category" in raws else continuous_raws[2]
        xcol=CONTINUOUS_DEFS[x_raw][0];zcol=CONTINUOUS_DEFS[z_raw][0]
        xgrid=np.linspace(df[x_raw].quantile(.03),df[x_raw].quantile(.97),75)
        zlevels=df[z_raw].quantile([.25,.75]).to_numpy()
        panel_levels=FACTORS["sex_category"] if panel_raw=="sex_category" else list(df[panel_raw].quantile([.25,.75]).to_numpy())
        fig,axs=plt.subplots(1,len(panel_levels),figsize=(6.8*len(panel_levels),5.3),squeeze=False)
        for ax,panel_value in zip(axs.flat,panel_levels):
            base=df.loc[df[panel_raw]==panel_value] if panel_raw=="sex_category" else df
            if panel_raw=="sex_category":
                sns.scatterplot(data=base,x=x_raw,y=col,hue=z_raw,palette="mako",s=16,alpha=.20,legend=False,ax=ax)
            else:
                sns.scatterplot(data=base,x=x_raw,y=col,s=14,alpha=.14,color="#667f8d",ax=ax)
            for j,z in enumerate(zlevels):
                grid=pd.DataFrame({xcol:(xgrid-df[x_raw].mean())/CONTINUOUS_DEFS[x_raw][2],
                    zcol:[(z-df[z_raw].mean())/CONTINUOUS_DEFS[z_raw][2]]*len(xgrid)})
                if panel_raw=="sex_category":
                    grid[panel_raw]=pd.Categorical([panel_value]*len(grid),categories=FACTORS["sex_category"])
                else:
                    panel_col=CONTINUOUS_DEFS[panel_raw][0]
                    grid[panel_col]=[(panel_value-df[panel_raw].mean())/CONTINUOUS_DEFS[panel_raw][2]]*len(grid)
                est=prediction_summary(fit, continuous_design_infos[(col,spec)], grid, alpha=.05)
                y=est["mean"].to_numpy();lo=est["mean_ci_lower"].to_numpy();hi=est["mean_ci_upper"].to_numpy()
                if SCALES[col]=="log10": y,lo,hi=10**y,10**lo,10**hi
                color=sns.color_palette("crest",2)[j]
                ax.plot(xgrid,y,lw=2.5,color=color,label=f"{CONTINUOUS_DEFS[z_raw][1]}={z:.1f}")
                ax.fill_between(xgrid,lo,hi,color=color,alpha=.13)
            panel_text=panel_value if panel_raw=="sex_category" else f"{CONTINUOUS_DEFS[panel_raw][1]}={panel_value:.1f}"
            ax.set(title=f"Conditioning level: {panel_text}",
                xlabel=f"{CONTINUOUS_DEFS[x_raw][1]} ({CONTINUOUS_DEFS[x_raw][3]})",
                ylabel=f"{label} ({unit})")
            if use_log_axis(col):ax.set_yscale("log")
            ax.legend(fontsize=9);sns.despine(ax=ax)
        r=continuous.loc[(continuous.hrv_variable==col)&(continuous.specification==spec)].iloc[0]
        fig.suptitle(f"{label} | {spec.replace('_',' × ')} | HC3 three-way interaction",fontsize=17,fontweight="semibold")
        fig.text(.5,.015,f"Three-way p={r.three_way_p_hc3:.3g}, BH q={r.three_way_q_bh_49:.3g} | partial R²={r.three_way_partial_r2:.3f}",
            ha="center",fontsize=10)
        fig.tight_layout(rect=(0,.05,1,.94),w_pad=1.9)
        savefig(fig,folder/f"07_continuous_{spec}.png")
print(f"Continuous/mixed figures saved: {len(HRV)*len(SPECS)}")

Continuous/mixed figures saved: 49


## 10. Global evidence synthesis and reproducibility checks

The atlas includes all 28 categorical and 49 continuous/mixed third-order results. Asterisk marks BH q < 0.05 **within the stated model family**. A summary table combines p/q, partial R², cell support, diagnostics and sensitivity. It is a navigation aid, not a ranking of clinical relevance or proof of causality.

In [11]:
for col,(slug,label,unit) in HRV.items():
    c=categorical.loc[categorical.hrv_variable==col].set_index("triplet").reindex([TRIPLET_NAMES[t] for t in TRIPLETS])
    cm=continuous.loc[continuous.hrv_variable==col].set_index("specification").reindex([s[0] for s in SPECS])
    fig,axes=plt.subplots(1,2,figsize=(15.5,5.2))
    sns.barplot(x=c.index,y=c.three_way_partial_r2,ax=axes[0],color="#3e8198")
    axes[0].set(title="Categorical three-factor interaction",xlabel="Factor triplet",ylabel="Incremental partial R²")
    axes[0].tick_params(axis="x",rotation=35)
    max_partial_r2=max(float(c.three_way_partial_r2.max()),1e-5)
    axes[0].set_ylim(0,max_partial_r2*1.30)
    for i,r in enumerate(c.itertuples()):
        axes[0].text(i,float(r.three_way_partial_r2)+max_partial_r2*.035,
            f"q={r.three_way_q_bh_28:.2g}",ha="center",va="bottom",fontsize=8.5)
    sns.barplot(x=cm.index,y=cm.three_way_partial_r2,ax=axes[1],color="#c58568")
    axes[1].set(title="Continuous/mixed three-factor interaction",xlabel="Specification",ylabel="Incremental partial R²")
    axes[1].tick_params(axis="x",rotation=40)
    fig.suptitle(f"{label} — trifactorial interaction evidence",fontsize=17,fontweight="semibold")
    fig.tight_layout(rect=(0,0,1,.94))
    savefig(fig,FIGURE_DIR/slug/"_summary"/"trifactorial_evidence.png")

for data,key,metric,qcolumn,title,outfile in [
    (categorical,"triplet","three_way_partial_r2","three_way_q_bh_28","Categorical three-way interaction","categorical_atlas.png"),
    (continuous,"specification","three_way_partial_r2","three_way_q_bh_49","Continuous/mixed three-way interaction","continuous_atlas.png"),
]:
    matrix=data.pivot(index="hrv_variable",columns=key,values=metric).reindex(index=list(HRV))
    qmat=data.pivot(index="hrv_variable",columns=key,values=qcolumn).reindex(index=matrix.index,columns=matrix.columns)
    ann=np.array([[f"{matrix.iloc[i,j]:.3f}"+("*" if qmat.iloc[i,j]<FDR_ALPHA else "")
        for j in range(matrix.shape[1])] for i in range(matrix.shape[0])])
    fig,ax=plt.subplots(figsize=(max(11,1.55*matrix.shape[1]+4),6.8))
    sns.heatmap(matrix,ax=ax,cmap="YlGnBu",annot=ann,fmt="",linewidths=.6,
        cbar_kws={"label":"Three-way incremental partial R²"})
    ax.set_yticklabels([HRV[k][1] for k in matrix.index],rotation=0)
    ax.tick_params(axis="x",rotation=35)
    ax.set(title=title+"  |  * = FDR q < 0.05",xlabel="Specification",ylabel="HRV metric")
    fig.tight_layout()
    savefig(fig,GLOBAL_DIR/outfile)

# Separate second-order interaction atlas for the hierarchical pair-interaction model.
second_pivot=second_effects.copy()
second_pivot["pair_term"]=second_pivot.factor_a.map(SHORT)+"×"+second_pivot.factor_b.map(SHORT)
second_pivot["test"]=second_pivot.triplet.str.replace("_x_","·",regex=False)+" / "+second_pivot.pair_term
matrix=second_pivot.pivot(index="hrv_variable",columns="test",values="q_bh_all_84_pair_tests").reindex(index=list(HRV))
fig,ax=plt.subplots(figsize=(max(16,1.1*matrix.shape[1]+4),6.8))
sns.heatmap(-np.log10(matrix.clip(lower=1e-16)),ax=ax,cmap="rocket_r",linewidths=.5,
            cbar_kws={"label":"−log10 BH-adjusted q-value (84 second-order tests)"})
ax.set_yticklabels([HRV[k][1] for k in matrix.index],rotation=0)
ax.tick_params(axis="x",rotation=45)
for tick in ax.get_xticklabels(): tick.set_ha("right")
ax.set(title="Second-order interactions adjusted for third factor | FDR-adjusted q-values",ylabel="",xlabel="")
fig.tight_layout()
savefig(fig,GLOBAL_DIR/"second_order_interaction_atlas.png")

summary=categorical[["hrv_variable","triplet","model_status","min_analysis_cell_n",
    "low_support_warning","three_way_p_hc3","three_way_q_bh_28","three_way_partial_r2",
    "full_model_r2","condition_number","max_leverage","max_cooks_distance"]].copy()
summary["eligible_nonparametric_contrasts"]=summary.apply(lambda r:int(
    ((conditional.hrv_variable==r.hrv_variable)&(conditional.triplet==r.triplet)&conditional.estimable).sum()),axis=1)
summary["potential_influence_caution"]=summary.max_cooks_distance > (4/len(df))
summary["max_leverage_caution"]=summary.max_leverage > (4*categorical.parameters.max()/len(df))
summary.to_csv(TABLE_DIR / "trifactorial_evidence_summary.csv",index=False)

# Strict consistency checks — outputs are rerunnable and no participants are filtered.
assert original_shape==(476,24)
assert len(HRV)==7 and len(TRIPLETS)==4
assert len(categorical)==28
assert len(main_effects)==84 and len(second_effects)==84
assert len(continuous)==49 and len(continuous_main)==147 and len(continuous_second)==147
assert categorical.model_status.eq("ok").all(),categorical.loc[categorical.model_status!="ok"]
assert continuous.model_status.eq("ok").all(),continuous.loc[continuous.model_status!="ok"]
assert len(conditional)==len(conditional_models)
assert len(support.query("category_scheme == 'analysis'"))==sum(np.prod([len(ANALYSIS_LEVELS[f]) for f in tri]) for tri in TRIPLETS)
assert support.query("category_scheme == 'analysis'").n.min()>=MIN_MODEL_CELL_N
assert not categorical.three_way_p_hc3.isna().any()
assert not continuous.three_way_p_hc3.isna().any()
assert len(list(FIGURE_DIR.rglob("01_original_distributions.png")))==28
assert len(list(FIGURE_DIR.rglob("02_original_ecdf_by_third_factor.png")))==28
assert len(list(FIGURE_DIR.rglob("03_conditioned_medians.png")))==28
assert len(list(FIGURE_DIR.rglob("04_hc3_three_way_interaction.png")))==28
assert len(list(FIGURE_DIR.rglob("05_conditional_cliffs_forest.png")))==28
assert len(list(FIGURE_DIR.rglob("06_model_diagnostics.png")))==28
assert len(list(FIGURE_DIR.rglob("07_continuous_*.png")))==49
assert len(list(FIGURE_DIR.rglob("trifactorial_evidence.png")))==7
assert len(list(GLOBAL_DIR.glob("*.png")))==7

print("Completed — 08 Trifactorial HRV analysis")
print(f"Unchanged cohort: {len(df)} participants")
print(f"Categorical models: {len(categorical)} | continuous/mixed models: {len(continuous)}")
print(f"Eligible conditional contrasts: {conditional.estimable.sum()}/{len(conditional)}")
print(f"HC3 model conditional contrasts: {len(conditional_models)}")
print(f"Sensitivity/feasibility records: {len(sensitivity)}")
print(f"CSV tables: {len(list(TABLE_DIR.glob('*.csv')))}")
print(f"PNG figures: {len(list(FIGURE_DIR.rglob('*.png')))}")
print(f"Results: {OUTPUT_DIR}")
display(summary.round(4))

Completed — 08 Trifactorial HRV analysis
Unchanged cohort: 476 participants
Categorical models: 28 | continuous/mixed models: 49
Eligible conditional contrasts: 903/966
HC3 model conditional contrasts: 966
Sensitivity/feasibility records: 77
CSV tables: 14
PNG figures: 231
Results: ../../eda_results/08_trifactorial_hrv_analysis


,hrv_variable,triplet,model_status,min_analysis_cell_n,low_support_warning,three_way_p_hc3,three_way_q_bh_28,three_way_partial_r2,full_model_r2,condition_number,max_leverage,max_cooks_distance,eligible_nonparametric_contrasts,potential_influence_caution,max_leverage_caution
0,t2m_pre_mean_rr,age_x_sex_x_bp,ok,13,False,0.6127,0.9099,0.0021,0.0371,3.7802,0.0769,0.0231,24,True,False
1,t2m_pre_mean_rr,age_x_sex_x_bmi,ok,11,False,0.1920,0.7680,0.0075,0.0344,3.8927,0.0909,0.0294,24,True,False
2,t2m_pre_mean_rr,age_x_bp_x_bmi,ok,12,False,0.5525,0.9099,0.0059,0.0470,4.3201,0.0833,0.0254,45,True,False
3,t2m_pre_mean_rr,sex_x_bp_x_bmi,ok,9,True,0.6341,0.9099,0.0055,0.0305,5.5744,0.1111,0.0291,36,True,False
4,t2m_pre_mean_hr,age_x_sex_x_bp,ok,13,False,0.6783,0.9099,0.0014,0.0336,3.7802,0.0769,0.0386,24,True,False
5,t2m_pre_mean_hr,age_x_sex_x_bmi,ok,11,False,0.1642,0.7680,0.0085,0.0320,3.8927,0.0909,0.0454,24,True,False
6,t2m_pre_mean_hr,age_x_bp_x_bmi,ok,12,False,0.4574,0.9099,0.0071,0.0468,4.3201,0.0833,0.0337,45,True,False
7,t2m_pre_mean_hr,sex_x_bp_x_bmi,ok,9,True,0.7292,0.9099,0.0048,0.0286,5.5744,0.1111,0.0542,36,True,False
8,t2m_pre_sdnn,age_x_sex_x_bp,ok,13,False,0.4702,0.9099,0.0042,0.1452,3.7802,0.0769,0.0305,24,True,False
9,t2m_pre_sdnn,age_x_sex_x_bmi,ok,11,False,0.4146,0.9099,0.0048,0.1431,3.8927,0.0909,0.0367,24,True,False
